<a href="https://colab.research.google.com/github/caralejandro12-gif/Estudio-de-planta-de-beneficio/blob/main/Preentrega1_Carabajal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Estudio de recuperacion metalurgica en planta de beneficio**

1. Problema de negocio

El circuito de molienda y flotación presenta un trade-off estructural entre recuperación metalúrgica y consumo energético específico (kWh/t), donde incrementos en la finura de molienda mejoran la liberación mineral pero generan aumentos no lineales en el consumo energético.

Este fenómeno introduce alta variabilidad operativa y dificulta la identificación del punto de operación óptimo, impactando directamente en la rentabilidad del proceso.

Actualmente, las decisiones sobre granulometría, clasificación y condiciones operativas se realizan de forma reactiva, sin integración sistemática de variables geológicas, mineralógicas y de proceso.

2. Marco operativo

Variables críticas para la calidad (CTQs):

Recuperación de Cu (%) → maximizar
Consumo energético específico (kWh/t) → minimizar

Desperdicios identificados:

- Sobre-molienda → consumo energético sin retorno metalúrgico
- Pérdida de metal en relaves → baja eficiencia de liberación o clasificación

Objetivo operacional:

- Reducir la variabilidad del proceso y estabilizar la operación en condiciones cercanas al óptimo técnico-económico.

3. Objetivo técnico

Desarrollar un modelo predictivo multisalida que permita estimar simultáneamente:
- Recuperación metalúrgica (%)
- Consumo energético (kWh/t)

y utilizarlo como base para evaluar escenarios operativos y asistir la toma de decisiones en tiempo casi real.

4. Función objetivo (CLAVE)

Se define la siguiente función de beneficio económico:

*BENEFICIO =(recuperacion x Ley x Tonelaje x PrecioCobre)-(Energia x Tonelaje x costoEnergia)*

**Suposición:
Se asume una ley de alimentación constante y precios promedio de mercado.**

5. Supuestos económicos
- Costo energía: 0.10 USD/kWh
- Precio Cu: 8500 USD/t
- Tonelaje: 10,000 tpd
- Ley alimentación: 0.6% Cu


## **Descripcion de variables**
df_metalurgico

- ID_Muestra: identificación de muestra geologica
- Zona: posicion de la muestra respecto al centro del yacimiento
- Coordenada_X: Latitud
- Coordenada_Y: Longitud
- Cota_Z: Altura
- Profundidad_Banco: idendificiacion del nivel
- Tipo_Mineral: Calsificación mineralogica
- Ley_Au_ppm: concentracion de metal de oro
- Ley_Ag_ppm: concentración de metal de plata
- Ley_Cu_%: concentración de metal de cobre
- Ley_Mo_%: Concentrcación de metal de molibdeno
- Ley_As_ppm: concentracion de arsenico
- Indice_Moliendabilidad_BWI: indice de moliendabilidad por tipo de mineral
- Dureza_Roca: resistencia mecanica uniaxial
- Densidad_ton_m3: densidad de roca
- Tipo_Roca: clasificacion de roca geomecanica
- Humedad_%: cantidad de agua que posee la roca

df_operacional

* Fecha: fecha de trabajo
* Turno: turno trabajado
* Flujo_Alimentacion_tph: tonelaje de mineral alimentado a planta
* Malla_Pasante_75um_%: porcentaje de mineral que pasa la malla 75micras
* Consumo_Acero_kg_t: consumo de bolas de acero y cubiertas.
* Flujo_Hidrociclon_Overflow_m3h: flujo de rebalse para flotacion
* Flujo_Hidrociclon_Underflow_m3h: flujo de abajo para recirculado
* Ley_Cabeza_Cu_%: concentracion de metal de alimentacion de cobre
* Ley_Cabeza_Au_ppm: concentracion de metal de alimentacion de oro
* Ley_Concentrado_Cu_%: concentracion de cobre recuperado para fundicion
- Ley_Colas_Cu_%: concentracion de cobre perdidas a diques
- Recuperacion_Cu_%: recuperacion de cobre respecto a la alimentacion
- Consumo_Energia_kWh_t: consumo de energia de los equipos de planta
- Consumo_Reactivo_kg_t: consumo de reactivos para flotacion
- Flujo_Bomba_Rougher_m3h: alimentacion de celdas de flotacion
- Flujo_Bomba_Scavenger_m3h: alimentacion de celdas de flotacion de colas
- Flujo_Bomba_Cleaner_m3h: alimentacion de celdas de recuperacion
- Colas_Finales_ton: total de desperdicio del proceso
- Turnos_Paradas: cantidad de paradas por turno

# **Librerias**

In [1]:
# Librerias necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from scipy import stats
from datetime import datetime

In [2]:
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)

In [4]:
url_metalurgico = "https://github.com/caralejandro12-gif/Estudio-de-planta-de-beneficio/releases/download/v1.0-Data/df_geometalurgico.csv"
url_operacional = "https://github.com/caralejandro12-gif/Estudio-de-planta-de-beneficio/releases/download/v1.0-Data/df_operacional.csv"

df_metalurgico = pd.read_csv(url_metalurgico)
df_operacional = pd.read_csv(url_operacional)

assert not df_metalurgico.empty, "Error: df_metalurgico vacío"
assert not df_operacional.empty, "Error: df_operacional vacío"